## About Guided Cursor

Welcome to **Guided Cursor**, an AI-powered platform designed to guide you through programming problems step by step. As with all AI systems, responses may not always be perfectly accurate. You are encouraged to think critically and verify all output independently.

**Privacy.** We may collect anonymised usage data to improve the platform and support academic research into AI-assisted pedagogy. No data will be shared outside the project team. Your usage and performance will **not** be disclosed to module leaders and will have **no bearing** on your academic grades.

**Data Retention.** This platform may be taken offline at the end of the academic term, and all stored data may be permanently deleted. Please back up any materials you wish to keep in advance.

**Contact.** For any questions or concerns, please reach out to **hello@guidedcursor.studio**.

# Root-Finding Methods

**Learning objectives**

By the end of this notebook you will be able to:

- Classify root-finding algorithms into bracketing and open methods
- Understand why the choice of rearrangement determines whether fixed-point iteration succeeds or fails
- Implement Newton's method for solving nonlinear equations
- Implement the Secant method as a derivative-free alternative
- Visualise Newton's fractal on the complex plane

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rootfinding_hints import show_hint
from rootfinding_verify import check_newton_method, check_secant_method

## 1. The Root-Finding Problem

Given a function $f(x)$, find $x^*$ such that $f(x^*) = 0$.

Root-finding algorithms generally fall into two families:

| Family | Examples | Pros | Cons |
|--------|----------|------|------|
| **Bracketing methods** | Bisection, Regula Falsi | Guaranteed to converge | Slow |
| **Open methods** | Fixed-point, Newton, Secant | Fast | May diverge |

**Bracketing methods** maintain an interval $[a, b]$ where $f(a)$ and $f(b)$ have opposite signs, then shrink the interval until the root is found. They are reliable but slow.

**Open methods** start from one or two initial guesses and generate a sequence of approximations. They can be much faster, but they offer no guarantee of convergence.

This notebook focuses on open methods. We will use a single equation throughout:

$$f(x) = x^3 - 1 = 0, \qquad x^* = 1$$

In [ ]:
# Complete code -- just run this cell
f = lambda x: x**3 - 1
df = lambda x: 3 * x**2
x_star = 1.0

x_plot = np.linspace(-0.5, 4, 300)
plt.figure(figsize=(7, 4))
plt.plot(x_plot, f(x_plot), color="steelblue", linewidth=2.5,
         label="$f(x) = x^3 - 1$")
plt.axhline(0, color="grey", linewidth=0.5)
plt.plot(x_star, 0, "*", color="black", markersize=10, zorder=5,
         label="Root $x^* = 1$")
plt.xlabel("x", fontsize=13)
plt.ylabel("f(x)", fontsize=13)
plt.ylim(-5, 50)
plt.title("Our running example: $f(x) = x^3 - 1$", fontsize=14)
plt.legend(fontsize=11)
plt.tick_params(labelsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Fixed-Point Iteration

The simplest open method. Rewrite $f(x) = 0$ as $x = \varphi(x)$ and iterate:

$$x_{k+1} = \varphi(x_k)$$

If the sequence converges, it converges to a **fixed point** where $x^* = \varphi(x^*)$, which is also a root of $f$.

The key question is: **how do we choose $\varphi$?** The same equation can be rearranged in many different ways, and the choice determines whether the iteration converges or diverges.

### The art of construction

For $x^3 - 1 = 0$, here are three different rearrangements:

| $\varphi(x)$ | Rearrangement | Behaviour |
|-------------|--------------|---|
| $x^3 + x - 1$ | $x = x^3 + x - 1$ | Diverges rapidly |
| $1 / x^2$ | $x = 1 / x^2$ | Oscillates and diverges |
| $(2x^3 + 1) / (3x^2)$ | $x = (2x^3 + 1) / (3x^2)$ | Converges quickly |

All three are valid rearrangements of the same equation, yet their behaviour could not be more different. The cobweb diagrams below make this visible.

In [ ]:
# Complete code -- just run this cell
def fixed_point_iteration(phi, x0, max_iter):
    """Run fixed-point iteration and return the history."""
    history = [x0]
    x = x0
    for _ in range(max_iter):
        x = phi(x)
        history.append(x)
        if abs(x) > 1e10:
            break
    return history

x0 = 0.5

# Three rearrangements of x^3 - 1 = 0
phi_a = lambda x: x**3 + x - 1
phi_b = lambda x: 1 / x**2
phi_c = lambda x: (2 * x**3 + 1) / (3 * x**2)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

schemes = [
    (phi_a, "$\\varphi(x) = x^3 + x - 1$",
     "$\\varphi(x) = x^3 + x - 1$ (diverges)",
     fixed_point_iteration(phi_a, x0, 3), (-1, 2.5)),
    (phi_b, "$\\varphi(x) = 1/x^2$",
     "$\\varphi(x) = 1/x^2$ (oscillates)",
     fixed_point_iteration(phi_b, x0, 3), (-1, 5)),
    (phi_c, "$\\varphi(x) = (2x^3+1)/(3x^2)$",
     "$\\varphi(x) = (2x^3+1)/(3x^2)$ (converges)",
     fixed_point_iteration(phi_c, x0, 8), (-0.5, 2.5)),
]

for ax, (phi, phi_label, title, hist, xlims) in zip(axes, schemes):
    x_range = np.linspace(xlims[0], xlims[1], 300)

    # Compute phi(x) with clipping for display
    phi_vals = np.array([phi(xi) if abs(xi) > 1e-6 else np.nan
                         for xi in x_range])
    phi_vals = np.clip(phi_vals, xlims[0] - 1, xlims[1] + 1)

    ax.plot(x_range, phi_vals, color="steelblue", linewidth=2.5,
            label=phi_label)
    ax.plot(x_range, x_range, "--", color="grey", linewidth=1,
            label="$y = x$")

    # Cobweb path
    cob_x, cob_y = [hist[0]], [hist[0]]
    for k in range(len(hist) - 1):
        cob_x.extend([hist[k], hist[k + 1]])
        cob_y.extend([hist[k + 1], hist[k + 1]])
    ax.plot(cob_x, cob_y, color="firebrick", linewidth=1.6, alpha=0.9)

    ax.plot(1, 1, "*", color="black", markersize=9, zorder=5,
            label="$x^* = 1$")

    ax.set_xlim(xlims)
    ax.set_ylim(xlims)
    ax.set_xlabel("x", fontsize=16)
    ax.set_ylabel("y", fontsize=16)
    ax.set_title(title, fontsize=15)
    ax.legend(fontsize=12, loc="upper left")
    ax.tick_params(labelsize=13)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The first two rearrangements spiral outward and never reach the root. The third spirals inward and converges quickly. In Section 3 we will see that this third rearrangement is actually Newton's method written in fixed-point form.

## 3. Newton's Method

### 3.1 Deriving the formula

Suppose we have a current guess $x_k$ that is close to the root but not exact. We want to find a better guess $x_{k+1}$.

At $x_k$, we know the value $f(x_k)$ and the slope $f'(x_k)$. From the definition of the derivative, a straight line through $(x_k, f(x_k))$ with slope $f'(x_k)$ is:

$$L(x) = f(x_k) + f'(x_k) \cdot (x - x_k)$$

This is the **tangent line** to $f$ at $x_k$. Near $x_k$ it closely approximates $f(x)$, so we set $L(x) = 0$ and solve for $x$:

$$f(x_k) + f'(x_k)(x - x_k) = 0 \quad \Longrightarrow \quad x = x_k - \frac{f(x_k)}{f'(x_k)}$$

This gives us **Newton's iteration formula**:

$$x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)}$$

At each step, we follow the tangent line down to the $x$-axis and use that crossing point as our next guess.

In [ ]:
# Complete code -- just run this cell
# Visualisation: two Newton steps on f(x) = x^3 - 1, starting from x0 = 3
x0_n = 3.0
x1_n = x0_n - f(x0_n) / df(x0_n)
x2_n = x1_n - f(x1_n) / df(x1_n)
pts_n = [x0_n, x1_n, x2_n]
colours_n = ["firebrick", "darkorange", "seagreen"]

x_plot = np.linspace(-0.5, 4, 300)

plt.figure(figsize=(9, 5.5))
plt.plot(x_plot, f(x_plot), color="steelblue", linewidth=2.5,
         label="$f(x) = x^3 - 1$")
plt.axhline(0, color="grey", linewidth=0.5)

# Tangent line at x0
tangent0 = f(x0_n) + df(x0_n) * (x_plot - x0_n)
plt.plot(x_plot, tangent0, "--", color=colours_n[0], linewidth=1.4,
         label="Tangent at $x_0$")

# Tangent line at x1
tangent1 = f(x1_n) + df(x1_n) * (x_plot - x1_n)
plt.plot(x_plot, tangent1, "--", color=colours_n[1], linewidth=1.4,
         label="Tangent at $x_1$")

# Mark each x_k on curve and x-axis with the same colour
for i, (xi, ci) in enumerate(zip(pts_n, colours_n)):
    if f(xi) <= 50:
        plt.plot(xi, f(xi), "o", color=ci, markersize=8)
    plt.plot(xi, 0, "v", color=ci, markersize=9,
             label=f"$x_{i} = {xi:.4f}$")

# Root
plt.plot(x_star, 0, "*", color="black", markersize=10, zorder=5,
         label="$x^* = 1$")

# Label x_k on the x-axis
for i, xi in enumerate(pts_n):
    plt.annotate(f"$x_{i}$", (xi, 0), textcoords="offset points",
                 xytext=(0, -15), ha="center", fontsize=11, color="black")

plt.ylim(-5, 50)
plt.xlabel("x", fontsize=13)
plt.ylabel("f(x)", fontsize=13)
plt.title("Newton's method: following tangent lines toward the root",
          fontsize=14)
plt.legend(fontsize=10)
plt.tick_params(labelsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2 Connection to fixed-point iteration

Newton's method can be written as a fixed-point iteration with

$$\varphi(x) = x - \frac{f(x)}{f'(x)}$$

For $f(x) = x^3 - 1$, this simplifies to

$$\varphi(x) = x - \frac{x^3 - 1}{3x^2} = \frac{2x^3 + 1}{3x^2}$$

This is exactly the third rearrangement from Section 2. Newton's method is not a separate idea; it is a systematic way to construct the right $\varphi$ for any smooth function.

### 3.3 Implementation

Algorithm:

1. Start from an initial guess $x_0$ and store it in a `history` list.
2. **Loop:** compute $x_{k+1} = x_k - g(x_k)/g'(x_k)$ and append to `history`.
3. Stop when $|g(x_k)| < \text{tol}$ or the iteration count reaches `max_iter`.
4. Return the final estimate and the full `history`.

Fill in the loop body below.

In [ ]:
def newton_method(g, dg, x0, tol=1e-10, max_iter=50):
    history = [x0]
    x = x0

    # ===== YOUR CODE BELOW =====
    for _ in range(max_iter):
        x = x - g(x) / dg(x)
        history.append(x)
        if abs(g(x)) < tol:
            break
    # ===== YOUR CODE ABOVE =====
    return x, history

In [ ]:
show_hint("newton_method")

In [ ]:
check_newton_method(newton_method)

### 3.4 Applying Newton's method to $x^3 - 1 = 0$

Starting from $x_0 = 3$, we apply Newton's method to our running example. Watch how the error (the distance from $x_k$ to the true root $x^* = 1$) shrinks rapidly with each iteration.

In [ ]:
# Complete code -- just run this cell
x_newton, hist_newton = newton_method(f, df, x0=3.0)

print(f"{'k':>3s}  {'x_k':>18s}  {'f(x_k)':>14s}  {'|x_k - x*|':>14s}")
print("-" * 54)
for k, xk in enumerate(hist_newton):
    print(f"{k:3d}  {xk:18.14f}  {f(xk):14.2e}  {abs(xk - x_star):14.2e}")

print(f"\nNewton's method converges in {len(hist_newton) - 1} iterations.")
print("Notice how the number of correct digits roughly doubles at each step.")

## 4. The Secant Method

### 4.1 Replacing the derivative with a slope

Newton's method needs $f'(x_k)$ at every step. If the derivative is expensive or unavailable, we can approximate it using the slope through the two most recent points:

$$f'(x_k) \approx \frac{f(x_k) - f(x_{k-1})}{x_k - x_{k-1}}$$

Substituting this into the Newton formula gives the **Secant method**:

$$x_{k+1} = x_k - f(x_k) \cdot \frac{x_k - x_{k-1}}{f(x_k) - f(x_{k-1})}$$

It requires **two** starting guesses ($x_0, x_1$) but no derivative.

In [ ]:
# Complete code -- just run this cell
# Visualisation: two Secant steps on f(x) = x^3 - 1
xs = [3.0, 2.0]  # x0, x1
for _ in range(2):  # compute x2, x3
    slope = (f(xs[-1]) - f(xs[-2])) / (xs[-1] - xs[-2])
    xs.append(xs[-1] - f(xs[-1]) / slope)

x_plot = np.linspace(-0.5, 4, 300)
colours_s = ["firebrick", "darkorange", "seagreen", "slateblue"]

plt.figure(figsize=(9, 5.5))
plt.plot(x_plot, f(x_plot), color="steelblue", linewidth=2.5,
         label="$f(x) = x^3 - 1$")
plt.axhline(0, color="grey", linewidth=0.5)

# Step 1: secant through x0 and x1
sl1 = (f(xs[1]) - f(xs[0])) / (xs[1] - xs[0])
sec1 = f(xs[0]) + sl1 * (x_plot - xs[0])
plt.plot(x_plot, sec1, "--", color="firebrick", linewidth=1.4, alpha=0.7,
         label="Secant step 1")

# Step 2: secant through x1 and x2
sl2 = (f(xs[2]) - f(xs[1])) / (xs[2] - xs[1])
sec2 = f(xs[1]) + sl2 * (x_plot - xs[1])
plt.plot(x_plot, sec2, "--", color="darkorange", linewidth=1.4, alpha=0.7,
         label="Secant step 2")

# Mark each x_k on the curve and x-axis, each with its own colour
for i, (xi, ci) in enumerate(zip(xs, colours_s)):
    fxi = f(xi)
    if abs(fxi) <= 50:
        plt.plot(xi, fxi, "o", color=ci, markersize=7)
    plt.plot(xi, 0, "v", color=ci, markersize=8,
             label=f"$x_{i} = {xi:.4f}$")

# Root
plt.plot(x_star, 0, "*", color="black", markersize=10, zorder=5,
         label="$x^* = 1$")

# Label x_k on the x-axis
for i, xi in enumerate(xs):
    plt.annotate(f"$x_{i}$", (xi, 0), textcoords="offset points",
                 xytext=(0, -15), ha="center", fontsize=11, color="black")

plt.ylim(-5, 50)
plt.xlabel("x", fontsize=13)
plt.ylabel("f(x)", fontsize=13)
plt.title("Secant method: two steps toward the root", fontsize=14)
plt.legend(fontsize=10)
plt.tick_params(labelsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.2 Implementation

Algorithm:

1. Start with two initial guesses $x_0, x_1$ and store both in `history`.
2. **Loop:** compute $x_{\text{new}} = x_1 - g(x_1) \cdot \frac{x_1 - x_0}{g(x_1) - g(x_0)}$, append to `history`, then shift $x_0, x_1 = x_1, x_{\text{new}}$.
3. Stop when $|g(x_1)| < \text{tol}$ or the iteration count reaches `max_iter`.
4. Return the final estimate and the full `history`.

Fill in the loop body below.

In [ ]:
def secant_method(g, x0, x1, tol=1e-10, max_iter=50):
    history = [x0, x1]

    # ===== YOUR CODE BELOW =====
    for _ in range(max_iter):
        g_x0, g_x1 = g(x0), g(x1)
        x_new = x1 - g_x1 * (x1 - x0) / (g_x1 - g_x0)
        history.append(x_new)
        x0, x1 = x1, x_new
        if abs(g(x1)) < tol:
            break
    # ===== YOUR CODE ABOVE =====
    return x1, history

In [ ]:
show_hint("secant_method")

In [ ]:
check_secant_method(secant_method)

### 4.3 Applying the Secant method to $x^3 - 1 = 0$

Starting from $x_0 = 3$ and $x_1 = 2$. The column $|x_k - x^*|$ shows the distance from the current estimate to the true root.

In [ ]:
# Complete code -- just run this cell
x_secant, hist_secant = secant_method(f, x0=3.0, x1=2.0)

print(f"{'k':>3s}  {'x_k':>18s}  {'f(x_k)':>14s}  {'|x_k - x*|':>14s}")
print("-" * 54)
for k, xk in enumerate(hist_secant):
    print(f"{k:3d}  {xk:18.14f}  {f(xk):14.2e}  {abs(xk - x_star):14.2e}")

print(f"\nSecant method converges in {len(hist_secant) - 2} iterations.")

## 5. Newton's Fractal

This section is inspired by the 3Blue1Brown video [*Newton's fractal (which Newton knew nothing about)*](https://www.youtube.com/watch?v=-RdOwhmqP5s), which beautifully illustrates the ideas below.

Newton's formula works just as well for **complex** numbers:

$$z_{k+1} = z_k - \frac{p(z_k)}{p'(z_k)}$$

Our equation $p(z) = z^3 - 1$ has three roots in the complex plane:

$$z_1 = 1, \qquad z_2 = e^{2\pi i/3}, \qquad z_3 = e^{4\pi i/3}$$

For each starting point $z_0$, Newton's method will eventually converge to **one** of the three roots. Colour each starting point according to which root it reaches, and a stunning **fractal** pattern appears at the boundaries.

Usually Newton's method converges to the nearest root, which seems reasonable. But at the boundaries between regions, the tiniest shift in starting position can send the iteration to a completely different root.

The boundary has a remarkable mathematical property: pick any point on it, and in any neighbourhood around it, no matter how small, there exist starting points that converge to **all three** roots.

In [ ]:
# Complete code -- just run this cell
res = 800
max_newt = 40
x_re = np.linspace(-2, 2, res)
x_im = np.linspace(-2, 2, res)
Re, Im = np.meshgrid(x_re, x_im)
Z = Re + 1j * Im

# Three roots of z^3 = 1
roots = np.array([1, np.exp(2j * np.pi / 3), np.exp(4j * np.pi / 3)])

# Newton iteration (vectorised)
for _ in range(max_newt):
    denom = 3 * Z**2
    denom = np.where(np.abs(denom) < 1e-12, 1e-12, denom)
    Z = Z - (Z**3 - 1) / denom

# Classify each point by which root it converged to
tol_conv = 1e-3
labels = np.full(Z.shape, 3, dtype=int)  # 3 = did not converge
for i, root in enumerate(roots):
    labels[np.abs(Z - root) < tol_conv] = i

from matplotlib.colors import ListedColormap
cmap = ListedColormap(["firebrick", "seagreen", "steelblue", "black"])

plt.figure(figsize=(8, 8))
plt.imshow(labels, extent=[-2, 2, -2, 2], cmap=cmap, origin="lower")
plt.xlabel("Re(z)", fontsize=13)
plt.ylabel("Im(z)", fontsize=13)
plt.title("Newton's fractal for $z^3 - 1 = 0$", fontsize=14)
plt.tick_params(labelsize=11)
plt.tight_layout()
plt.show()

## Closing Remarks

Newton published his iterative method in 1669, three centuries before Mandelbrot coined the word *fractal*. He had no idea that his simple formula for solving equations would produce such breathtaking patterns in the complex plane.

This is a recurring theme in the history of mathematics:

- **Newton** knew nothing about fractals. He simply wanted a better way to solve equations.
- **Hamilton** invented quaternions in 1843 out of pure algebraic curiosity. Over a century later, they became a cornerstone of quantum mechanics.
- **Fourier** decomposed heat conduction into sine waves in 1807. The Fast Fourier Transform was not discovered until 1965, and that is exactly the topic of our next chapter.
- Today, simple gradient descent and matrix multiplication have given rise to machine learning and large language models, the very tool you are using right now.

Simple ideas from centuries ago continue to seed entirely new discoveries. Perhaps there are questions yet to be asked, waiting for you to ask and answer them.

### Summary

| Method | Needs derivative? | Convergence | Key property |
|--------|:-:|---|---|
| Fixed-point | No | Depends entirely on $\varphi$ | Same equation, different $\varphi$ $\Rightarrow$ different divergence |
| Newton | Yes | Quadratic (order 2) | Automatically constructs a good $\varphi$ from $f$ and $f'$ |
| Secant | No | Super-linear (order $\approx 1.618$) | Approximates Newton without computing derivatives |

**Key takeaways:**

- Root-finding methods split into bracketing (reliable, slow) and open (fast, may diverge).
- Fixed-point iteration is simple but fragile: its convergence speed depends entirely on how you rewrite $f(x) = 0$ as $x = \varphi(x)$. A clever choice converges quickly; a poor choice diverges.
- Newton's method provides a principled way to choose $\varphi$, giving it fast (quadratic) convergence near the root.
- The Secant method approximates Newton's method without needing derivatives, at the cost of a slightly lower convergence rate.
- Extending Newton's method to the complex plane produces Newton's fractal, a simple algorithm containing infinite complexity.